In [0]:
from pyspark.sql import functions as F

In [0]:
raw_path = "/Volumes/workspace/default/books_raw/books_enriched.csv"
target_table = "workspace.default.books_simple"

In [0]:
#read raw csv
def read_books_csv(path):
    df = spark.read.option("header", "true").option("inferSchema", "true").option("multiline", "true").option("quote", '"').option("escape", '"').csv(path)
    print(f"Read {df.count()} rows from {path}")
    return df

In [0]:
#check file if has the columns 
def check_columns_exist(df, expected_columns):
    missing = set(expected_columns) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    print("All expected columns are present.")

In [0]:
#clean data
def clean_books(df):
    df = df.filter(df.book_id.isNotNull())
    df = df.withColumn("pages", F.col("pages").cast("double").cast("int"))
    df = df.withColumn("original_publication_year", F.col("original_publication_year").cast("double").cast("int"))
    return df
    

In [0]:
#ex a quality check for nulls
def check_nulls_in_key(df, key_column):
    null_count = df.filter(df[key_column].isNull()).count()
    if null_count > 0:
        raise ValueError(f"Found {null_count} rows with a mising {key_column}")
    print(f"No nulls found in {key_column}")

In [0]:
#check duplicates
def check_duplicates_in_key(df, key_column):
    total_rows = df.count()
    #added after the testing
    df_unique = df.dropDuplicates([key_column])

    unique_rows = df.dropDuplicates([key_column]).count()
    duplicate_count = total_rows - unique_rows

    if duplicate_count > 0:
        print(f"Found {duplicate_count} duplicate values in {key_column}")
    else:
        print(f"No duplicates found in {key_column}")
    return df_unique

In [0]:
#save the result, merge
from delta.tables import DeltaTable

def save_table_merge(df, table_name, key_column):
    if not spark.catalog.tableExists(table_name):
        #if a table doesn't exist, it will be created
        df.write.format("delta").saveAsTable(table_name)
        print(f"Created {table_name} with {df.count()} rows")
        return
    
    #if the table exist, merge
    target = DeltaTable.forName(spark, table_name)
    (
        target.alias("t").merge(df.alias("s"), f"t.{key_column} = s.{key_column}").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    )

    print(f"Merge {df.count} rows into {table_name}")


In [0]:
#testing
books_df = read_books_csv(raw_path)

Read 10000 rows from /Volumes/workspace/default/books_raw/books_enriched.csv


In [0]:
check_columns_exist(books_df, ["book_id", "title", "authors", "average_rating", "pages"])

All expected columns are present.


In [0]:
books_df = clean_books(books_df)

In [0]:
check_nulls_in_key(books_df, "book_id")

No nulls found in book_id


In [0]:
#found null values in testing
#in clean_books, i add:
#df = df.filter(df.book_id.isNotNull()) ---> remove rows without book_id

In [0]:
books_df = check_duplicates_in_key(books_df, "book_id")

No duplicates found in book_id


In [0]:
#testing the function to see if the duplicates are removed
total = books_df.count()
unique = books_df.dropDuplicates(["book_id"]).count()
print(f"Total rows {total}, unique book_id {unique}")

Total rows 10000, unique book_id 10000


In [0]:
#found 23 duplicates in testing
# i add in check_duplicates_in_key: 
# 1. df_unique = df.dropDuplicates([key_column])
# 2. remove raise ValueError so the pipline can continue (raise ValueError(f"Found {duplicate_count} duplicate values in {key_column}")) with print(f"Warning: found {duplicate_count} duplicate values in {key_column}")
# 3. function return df_unique, the data without duplicates

In [0]:
save_table_merge(books_df, target_table, key_column="book_id")

Created workspace.default.books_simple with 10000 rows


In [0]:
#error: The value '1933.0' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018

# why: original_year_publication column has 193.0 value, this is float. we try to cast  directly to int and failed, because Spark has s strict mode. 

# correction: cast in two steps: double and then int

In [0]:
# error 2: The value ' Ellis creates a character to whom North American children will have no difficulty relating. The daughter of university-educated parents' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018

# why: wrong way of reading the CSV. it's seems that description column misaligned with original_publication_year 
# correction: in read_book_csv, i add: multineline=true, quote='"' and escape= '"'

In [0]:
print("ETL finished.")

ETL finished.
